# Loading the data from landing folder to Bronze

In [0]:
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

In [0]:
dbutils.widgets.text("table_name", "customers")
table_name = dbutils.widgets.get("table_name")

In [0]:

customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_unique_id", StringType(), True),
    StructField("customer_zip_code_prefix", IntegerType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True),
])

payments_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("payment_sequential", IntegerType(), True),
    StructField("payment_type", StringType(), True),
    StructField("payment_installments", IntegerType(), True),
    StructField("payment_value", DoubleType(), True),
])

reviews_schema = StructType([
    StructField("review_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("review_score", IntegerType(), True),
    StructField("review_comment_title", StringType(), True),
    StructField("review_comment_message", StringType(), True),
    StructField("review_creation_date", TimestampType(), True),
    StructField("review_answer_timestamp", TimestampType(), True),
])

order_items_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_item_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", TimestampType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True),
])

sellers_schema = StructType([
    StructField("seller_id", StringType(), True),
    StructField("seller_zip_code_prefix", IntegerType(), True),
    StructField("seller_city", StringType(), True),
    StructField("seller_state", StringType(), True),
])

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", TimestampType(), True),
    StructField("order_approved_at", TimestampType(), True),
    StructField("order_delivered_carrier_date", TimestampType(), True),
    StructField("order_delivered_customer_date", TimestampType(), True),
    StructField("order_estimated_delivery_date", TimestampType(), True),
    StructField("processing_time", TimestampType(), True),
])

products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_category_name", StringType(), True),
    StructField("product_name_lenght", IntegerType(), True),
    StructField("product_description_lenght", IntegerType(), True),
    StructField("product_photos_qty", IntegerType(), True),
    StructField("product_weight_g", DoubleType(), True),
    StructField("product_length_cm", DoubleType(), True),
    StructField("product_height_cm", DoubleType(), True),
    StructField("product_width_cm", DoubleType(), True),
])

schema_map = {
    "customers": customers_schema,
    "payments": payments_schema,
    "reviews": reviews_schema,
    "order_items": order_items_schema,
    "sellers": sellers_schema,
    "orders": orders_schema,
    "products": products_schema,
}

if table_name not in schema_map:
      raise ValueError(f"No schema defined for table: {table_name}")

In [0]:
df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .schema(schema_map[table_name])
    .option("header", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .load(f"abfss://project01-platform@storageproject01.dfs.core.windows.net/landing-project01/{table_name}/")
    .withColumn("processing_time", current_timestamp())
    .drop("_rescued_data")
)

(df.writeStream
    .format("delta")
    .option("checkpointLocation", f"/Volumes/project01_databricks01/bronze/autoloader_checkpoint/{table_name}/")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"project01_databricks01.bronze.{table_name}")
)